Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import json, math, random

Reload vocab + data (self-contained notebook)

In [3]:
with open('../src/vocab.json') as f:
    vocab = json.load(f)
stoi = vocab['stoi']
itos = {int(k): v for k, v in vocab['itos'].items()}
PAD, BOS, EOS = 0, 1, 2
vocab_size = 39

paths, labels = [], []
with open("../data/synth/train_v2/labels.txt") as f:
    for line in f:
        fname, label = line.strip().split("\t")
        paths.append(f"../data/synth/train_v2/{fname}")
        labels.append(label)

transform = T.Compose([
    T.Resize((32, 128)), T.Grayscale(), T.ToTensor(), T.Normalize([0.5], [0.5]),
])

class OCRDataset(Dataset):
    def __init__(self, paths, labels, transform, stoi):
        self.paths, self.labels, self.transform, self.stoi = paths, labels, transform, stoi
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = self.transform(Image.open(self.paths[idx]).convert('RGB'))
        ids = [self.stoi[c] for c in self.labels[idx].lower() if c in self.stoi]
        return img, ids

def collate_fn(batch):
    imgs, label_lists = zip(*batch)
    imgs = torch.stack(imgs)
    max_len = max(len(l) for l in label_lists) + 1
    tgt_in = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    tgt_out = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    for i, ids in enumerate(label_lists):
        seq = [BOS] + ids
        tgt_in[i, :len(seq)] = torch.tensor(seq)
        out = ids + [EOS]
        tgt_out[i, :len(out)] = torch.tensor(out)
    return imgs, tgt_in, tgt_out

Paste in CNNEncoder and Decoder classes

In [4]:
class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(), nn.MaxPool2d((2, 1)),
        )
    def forward(self, x):
        x = self.conv(x)
        x = x.squeeze(2)
        return x.permute(0, 2, 1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.dk = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, q_in, kv_in, mask=None):
        B, Tq, _ = q_in.shape
        Tk = kv_in.shape[1]
        Q = self.q_proj(q_in).view(B, Tq, self.h, self.dk).transpose(1, 2)
        K = self.k_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        V = self.v_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.dk)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = scores.softmax(dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).contiguous().view(B, Tq, -1)
        return self.out_proj(out)

def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(), nn.Linear(ff_dim, d_model))
    def forward(self, x):
        return self.net(x)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, ff_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    def forward(self, x, memory, self_mask):
        normed = self.norm1(x)
        x = x + self.self_attn(normed, normed, mask=self_mask)
        x = x + self.cross_attn(self.norm2(x), memory, mask=None)
        x = x + self.ffn(self.norm3(x))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024, max_len=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, ff_dim) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
    def forward(self, tgt_in, memory):
        B, T = tgt_in.shape
        x = self.embed(tgt_in) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        mask = causal_mask(T, tgt_in.device)
        for layer in self.layers:
            x = layer(x, memory, mask)
        x = self.norm_out(x)
        return self.fc_out(x)

Combine into one model

In [5]:
class OCRModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024):
        super().__init__()
        self.encoder = CNNEncoder(d_model)
        self.decoder = Decoder(vocab_size, d_model, n_heads, n_layers, ff_dim)

    def forward(self, imgs, tgt_in):
        memory = self.encoder(imgs)
        return self.decoder(tgt_in, memory)

Set up device, model, optimizer

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

model = OCRModel(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} params")

cuda
4,732,583 params


Loss at step 0 sanity check

In [7]:
ds = OCRDataset(paths, labels, transform, stoi)
loader = DataLoader(ds, batch_size=32, collate_fn=collate_fn, shuffle=True)

imgs, tgt_in, tgt_out = next(iter(loader))
imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)

with torch.no_grad():
    logits = model(imgs, tgt_in)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1), ignore_index=PAD)
print(loss.item())   # expect ~3.66

3.8596408367156982


THE critical check - overfit 32 examples

In [8]:
imgs32, tgt_in32, tgt_out32 = imgs[:32].to(device), tgt_in[:32].to(device), tgt_out[:32].to(device)

model.train()
for step in range(500):
    optimizer.zero_grad()
    logits = model(imgs32, tgt_in32)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out32.reshape(-1), ignore_index=PAD)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(step, loss.item())

0 3.843471050262451
50 0.7520502209663391
100 0.06303778290748596
150 0.034955065697431564
200 0.020810386165976524
250 0.01533057913184166
300 0.009329475462436676
350 0.009503812529146671
400 0.002666286425665021
450 0.0018522674217820168


Reset the model fresh

In [9]:
model = OCRModel(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

LR warmup scheduler

In [10]:
from torch.optim.lr_scheduler import LambdaLR

warmup_steps = 500   
def lr_lambda(step):
    return min((step + 1) / warmup_steps, 1.0)

scheduler = LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler()

Train/val split

In [8]:
n = len(ds)
val_size = int(0.1 * n)
train_ds, val_ds = torch.utils.data.random_split(ds, [n - val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=128, collate_fn=collate_fn, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=128, collate_fn=collate_fn, shuffle=False, num_workers=0)

Time one epoch first

In [12]:
import time

model.train()
start = time.time()
total_loss = 0
for imgs, tgt_in, tgt_out in train_loader:
    imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)
    optimizer.zero_grad()

    with torch.amp.autocast(device_type='cuda'):
        logits = model(imgs, tgt_in)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1),
                                ignore_index=PAD, label_smoothing=0.1)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    total_loss += loss.item()

elapsed = time.time() - start
print(f"epoch 0: loss {total_loss/len(train_loader):.4f}  time {elapsed:.1f}s")

epoch 0: loss 3.1363  time 54.6s


Actual training loop

In [9]:
def evaluate_loss(model, loader):
    model.eval()
    total = 0
    with torch.no_grad():
        for imgs, tgt_in, tgt_out in loader:
            imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)
            with torch.amp.autocast(device_type='cuda'):
                logits = model(imgs, tgt_in)
                loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1), ignore_index=PAD)
            total += loss.item()
    model.train()
    return total / len(loader)


In [ ]:

model = OCRModel(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = LambdaLR(optimizer, lambda step: min((step+1)/warmup_steps, 1.0))
scaler = torch.amp.GradScaler()

epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for imgs, tgt_in, tgt_out in train_loader:
        imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            logits = model(imgs, tgt_in)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1),
                                    ignore_index=PAD, label_smoothing=0.1)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate_loss(model, val_loader)   # reuse from earlier — make sure it's defined in this notebook
    print(f"epoch {epoch}: train {train_loss:.4f}  val {val_loss:.4f}")

In [15]:
torch.save(model.state_dict(), '../checkpoints/model_50k.pt')

Training loop v2

In [10]:
model = OCRModel(vocab_size=vocab_size).to(device)   # fresh random weights
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

from torch.optim.lr_scheduler import CosineAnnealingLR
epochs = 30
scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
scaler = torch.amp.GradScaler()

# sanity check — MUST print ~3.66 before you proceed
with torch.no_grad():
    imgs, tgt_in, tgt_out = next(iter(train_loader))
    imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)
    logits = model(imgs, tgt_in)
    loss0 = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1), ignore_index=PAD)
print("pre-training loss check:", loss0.item())

pre-training loss check: 3.831632137298584


In [11]:
best_val_loss = float('inf')
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for imgs, tgt_in, tgt_out in train_loader:
        imgs, tgt_in, tgt_out = imgs.to(device), tgt_in.to(device), tgt_out.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            logits = model(imgs, tgt_in)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1),
                                    ignore_index=PAD, label_smoothing=0.1)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate_loss(model, val_loader)
    print(f"epoch {epoch}: train {train_loss:.4f}  val {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../checkpoints/model_50k_v2_best.pt')

epoch 0: train 2.6844  val 1.9296
epoch 1: train 1.3693  val 0.4675
epoch 2: train 0.8924  val 0.2652
epoch 3: train 0.7932  val 0.2145
epoch 4: train 0.7584  val 0.1911
epoch 5: train 0.7373  val 0.1845
epoch 6: train 0.7262  val 0.1697
epoch 7: train 0.7157  val 0.1614
epoch 8: train 0.7101  val 0.1611
epoch 9: train 0.7027  val 0.1581
epoch 10: train 0.6999  val 0.1649
epoch 11: train 0.6953  val 0.1452
epoch 12: train 0.6917  val 0.1422
epoch 13: train 0.6874  val 0.1424
epoch 14: train 0.6857  val 0.1399
epoch 15: train 0.6858  val 0.1411
epoch 16: train 0.6828  val 0.1353
epoch 17: train 0.6806  val 0.1416
epoch 18: train 0.6789  val 0.1346
epoch 19: train 0.6782  val 0.1393
epoch 20: train 0.6774  val 0.1321
epoch 21: train 0.6770  val 0.1316
epoch 22: train 0.6763  val 0.1310
epoch 23: train 0.6759  val 0.1313
epoch 24: train 0.6757  val 0.1310
epoch 25: train 0.6756  val 0.1311
epoch 26: train 0.6755  val 0.1310
epoch 27: train 0.6755  val 0.1311
epoch 28: train 0.6755  val 0.